# WEEK:  WEEK -09 DECISION TREE [C4.5 AND CART] 
 Chirag Rao K V

 240962180
 
 AIML B 16

In [31]:
import pandas as pd
import numpy as np
from collections import Counter
import math
from collections import Counter
import math
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text


In [32]:
d = [
    ["Sunny", 85, 85, "Weak", "No"],
    ["Sunny", 80, 90, "Strong", "No"],
    ["Overcast", 83, 78, "Weak", "Yes"],
    ["Rain", 70, 96, "Weak", "Yes"],
    ["Rain", 68, 80, "Weak", "Yes"],
    ["Rain", 65, 70, "Strong", "No"],
    ["Overcast", 64, 65, "Strong", "Yes"],
    ["Sunny", 72, 95, "Weak", "No"],
    ["Sunny", 69, 70, "Weak", "Yes"],
    ["Rain", 75, 80, "Weak", "Yes"],
    ["Sunny", 75, 70, "Strong", "Yes"],
    ["Overcast", 72, 90, "Strong", "Yes"],
    ["Overcast", 81, 75, "Weak", "Yes"],
    ["Rain", 71, 80, "Strong", "No"],
]


def c45_gr(col):
  y = [r[-1] for r in d]
  base = (
      -sum(
          (c / len(y)) * math.log2(c / len(y)) for c in Counter(y).values()
      )
      if y
      else 0
  )
  n, vals = len(d), set(r[col] for r in d)
  is_str = isinstance(next(iter(vals)), str)

  candidates = (
      [[[r for r in d if r[col] == v] for v in vals]]
      if is_str
      else [
          [[r for r in d if r[col] <= t], [r for r in d if r[col] > t]]
          for t in sorted(vals)[:-1]
      ]
  )

  best_gr = 0
  for splits in candidates:
    splits = [s for s in splits if s]
    new_e = 0
    for s in splits:
      sy = [r[-1] for r in s]
      e = (
          -sum(
              (c / len(sy)) * math.log2(c / len(sy))
              for c in Counter(sy).values()
          )
          if sy
          else 0
      )
      new_e += (len(s) / n) * e
    si = -sum((len(s) / n) * math.log2(len(s) / n) for s in splits if len(s))
    gr = (base - new_e) / si if si > 0 else 0
    best_gr = max(best_gr, gr)
  return best_gr


for i, name in enumerate(["Outlook", "Temp", "Humidity", "Wind"]):
  print(f"{name} Gain Ratio: {c45_gr(i):.4f}")

Outlook Gain Ratio: 0.1564
Temp Gain Ratio: 0.3055
Humidity Gain Ratio: 0.1285
Wind Gain Ratio: 0.0488


In [33]:
def classify(sample):
  outlook, temp, humidity, wind = sample

  if outlook == "Overcast":
    return "Yes"
  elif outlook == "Rain":
    return "Yes" if wind == "Weak" else "No"
  elif outlook == "Sunny":
    return "Yes" if humidity <= 75 else "No"


# Test with a new sample: [Outlook, Temp, Humidity, Wind]
new_sample = ["Sunny", 69, 70, "Weak"]
print(f"Sample: {new_sample}")
print(f"Play tennis: {classify(new_sample)}")

Sample: ['Sunny', 69, 70, 'Weak']
Play tennis: Yes


Q2

In [34]:
gi = lambda y: (
    1 - sum((c / len(y)) ** 2 for c in Counter(y).values()) if y else 0
)


def cart_gini(col):
  n, vals = len(d), set(r[col] for r in d)
  is_str = isinstance(next(iter(vals)), str)
  splits_list = (
      [[[r for r in d if r[col] == v], [r for r in d if r[col] != v]] for v in vals]
      if is_str
      else [
          [[r for r in d if r[col] <= t], [r for r in d if r[col] > t]]
          for t in sorted(vals)[:-1]
      ]
  )
  return min(
      sum(
          (len(s) / n) * gi([r[-1] for r in s]) for s in splits if len(s) > 0
      )
      for splits in splits_list
  )


for i, name in enumerate(["Outlook", "Temp", "Humidity", "Wind"]):
  print(f"{name} Gini Impurity: {cart_gini(i):.4f}")


def classify(sample):
  outlook, temp, humidity, wind = sample
  if outlook == "Overcast":
    return "Yes"
  return "No" if humidity > 75 or (outlook == "Rain" and wind == "Strong") else "Yes"


print("Classification:", classify(["Sunny", 72, 85, "Weak"]))

Outlook Gini Impurity: 0.3571
Temp Gini Impurity: 0.3956
Humidity Gini Impurity: 0.3937
Wind Gini Impurity: 0.4286
Classification: No


In [38]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

# Dataset
d = [
    ["Low", "Good", "Yes"],
    ["Low", "Bad", "No"],
    ["Medium", "Good", "Yes"],
    ["Medium", "Bad", "Yes"],
    ["High", "Good", "Yes"],
    ["High", "Bad", "No"],
]

df = pd.DataFrame(d, columns=["Income", "Credit", "Loan"])
enc = OrdinalEncoder()
X = enc.fit_transform(df[["Income", "Credit"]])
y = df["Loan"]
c45 = DecisionTreeClassifier(criterion="entropy").fit(X, y)
cart = DecisionTreeClassifier(criterion="gini").fit(X, y)
print("--- C4.5 (Entropy) Decision Tree ---")
print(export_text(c45, feature_names=["Income", "Credit"]))
print("--- CART (Gini) Decision Tree ---")
print(export_text(cart, feature_names=["Income", "Credit"]))
new_sample = enc.transform([["Low", "Good"]])
print("C4.5 (Entropy) Prediction:", c45.predict(new_sample)[0])
print("CART (Gini) Prediction:", cart.predict(new_sample)[0])
print("without sikitlearn")
def build(data, f):
  y = [r[-1] for r in data]
  if len(set(y)) == 1 or not f:
    return Counter(y).most_common(1)[0][0]
  base = -sum(
      (c / len(data)) * math.log2(c / len(data)) for c in Counter(y).values()
  )
  best = max(
      range(len(f)),
      key=lambda i: base
      - sum(
          (len(s) / len(data))
          * -sum(
              (c / len(s)) * math.log2(c / len(s))
              for c in Counter(r[-1] for r in s).values()
          )
          for v in set(r[i] for r in data)
          if (s := [r for r in data if r[i] == v])
      ),
  )
  return {
      f[best]: {
          v: build(
              [r[:best] + r[best + 1 :] for r in data if r[best] == v],
              f[:best] + f[best + 1 :],
          )
          for v in set(r[best] for r in data)
      }
  }


# Build & Print Tree
tree = build(d, ["Income", "Credit"])
print("Tree Structure:", tree)

# Classify New Sample
classify = (
    lambda s, t: t
    if not isinstance(t, dict)
    else classify(
        s, t[k := list(t.keys())[0]][s[["Income", "Credit"].index(k)]]
    )
)
print("Prediction for ['Low', 'Good']:", classify(["Low", "Good"], tree))

--- C4.5 (Entropy) Decision Tree ---
|--- Credit <= 0.50
|   |--- Income <= 1.50
|   |   |--- class: No
|   |--- Income >  1.50
|   |   |--- class: Yes
|--- Credit >  0.50
|   |--- class: Yes

--- CART (Gini) Decision Tree ---
|--- Credit <= 0.50
|   |--- Income <= 1.50
|   |   |--- class: No
|   |--- Income >  1.50
|   |   |--- class: Yes
|--- Credit >  0.50
|   |--- class: Yes

C4.5 (Entropy) Prediction: Yes
CART (Gini) Prediction: Yes
without sikitlearn
Tree Structure: {'Credit': {'Good': 'Yes', 'Bad': {'Income': {'Medium': 'Yes', 'High': 'No', 'Low': 'No'}}}}
Prediction for ['Low', 'Good']: Yes


/home/student/aiml_b/myenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


# Additional Questins 

In [42]:

df = pd.DataFrame({
    'Outlook': [
        'Sunny',
        'Sunny',
        'Overcast',
        'Rain',
        'Rain',
        'Rain',
        'Overcast',
        'Sunny',
        'Sunny',
        'Rain',
        'Sunny',
        'Overcast',
        'Overcast',
        'Rain',
    ],
    'Temp': [85, 80, 83, 70, 68, 65, 64, 72, 69, 75, 75, 72, 81, 71],
    'Humidity': [85, 90, 78, 96, 80, 70, 65, 95, 70, 80, 70, 90, 75, 80],
    'Wind': [
        'Weak',
        'Strong',
        'Weak',
        'Weak',
        'Weak',
        'Strong',
        'Strong',
        'Weak',
        'Weak',
        'Weak',
        'Strong',
        'Strong',
        'Weak',
        'Strong',
    ],
    'PlayTennis': [
        'No',
        'No',
        'Yes',
        'Yes',
        'Yes',
        'No',
        'Yes',
        'No',
        'Yes',
        'Yes',
        'Yes',
        'Yes',
        'Yes',
        'No',
    ],
})
enc = OrdinalEncoder()
X = enc.fit_transform(df[['Outlook', 'Temp', 'Humidity', 'Wind']])
y = df['PlayTennis']
c45_model = DecisionTreeClassifier(criterion='entropy').fit(X, y)
print('--- C4.5 Tree Rules ---')
print(export_text(c45_model, feature_names=['Outlook', 'Temp', 'Humidity', 'Wind']))
new_sample = enc.transform([['Sunny', 72, 85, 'Weak']])
print('Prediction for new sample:', c45_model.predict(new_sample)[0])

--- C4.5 Tree Rules ---
|--- Outlook <= 0.50
|   |--- class: Yes
|--- Outlook >  0.50
|   |--- Temp <= 7.50
|   |   |--- Temp <= 6.50
|   |   |   |--- Temp <= 4.50
|   |   |   |   |--- Wind <= 0.50
|   |   |   |   |   |--- class: No
|   |   |   |   |--- Wind >  0.50
|   |   |   |   |   |--- class: Yes
|   |   |   |--- Temp >  4.50
|   |   |   |   |--- class: No
|   |   |--- Temp >  6.50
|   |   |   |--- class: Yes
|   |--- Temp >  7.50
|   |   |--- class: No

Prediction for new sample: No


/home/student/aiml_b/myenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


In [44]:
cart_model = DecisionTreeClassifier(criterion='gini').fit(X, y)

print('\n--- CART Tree Rules ---')
print(export_text(
    cart_model, feature_names=['Outlook', 'Temp', 'Humidity', 'Wind']
))
print('Prediction for new sample:', cart_model.predict(new_sample)[0])


--- CART Tree Rules ---
|--- Outlook <= 0.50
|   |--- class: Yes
|--- Outlook >  0.50
|   |--- Temp <= 7.50
|   |   |--- Temp <= 1.50
|   |   |   |--- class: No
|   |   |--- Temp >  1.50
|   |   |   |--- Temp <= 4.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Temp >  4.50
|   |   |   |   |--- Temp <= 6.50
|   |   |   |   |   |--- class: No
|   |   |   |   |--- Temp >  6.50
|   |   |   |   |   |--- class: Yes
|   |--- Temp >  7.50
|   |   |--- class: No

Prediction for new sample: No
